<a href="https://colab.research.google.com/github/Chanthul4054/E-Policing-An-Integrated-Spatio-Temporal-Crime-Forecasting-and-Decision-Support-System/blob/Spatio-Temporal-Crime-Prediction-System/Component_1_Model_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Selecting a Suitable Model

Model Benchmarking

Hyperparameter Tuning

Download/Export the Model

# Selecting a Suitable Model

#CRIME HOTSPOT PREDICTION - model predicts a single "Total Risk" score

## How the "One Instance" Workflow WorksWhen a user selects a crime type (e.g., "Theft") on your dashboard, the system follows this logic:Target Selection: The model focuses only on "Theft" records in your historical data.Probability Calculation: It calculates the "intensity" (0% to 100%) for every GN Division specifically for that crime.Color Mapping: The GN divisions are then colored on your heatmap:Dark Red: High Intensity ($>80\%$ probability of Theft).Yellow/Orange: Moderate Intensity ($40-60\%$).Green: Low Intensity ($<10\%$).

#Most reliable metrices
- precision
- Recall
- F1-Score
- AUC-ROC

Load the training data

In [111]:
import pandas as pd
import joblib
import json
import os
from google.colab import drive
import numpy as np

# Mount Google Drive
drive.mount('/content/drive')
save_path = '/content/drive/MyDrive/DSGP/'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [112]:
import time
from sklearn.metrics import (classification_report, roc_auc_score, precision_score,
                            recall_score, f1_score, confusion_matrix)

# Load the full processed sets (Parquet preserves data types)
train_df = pd.read_parquet(os.path.join(save_path, 'train_set_processed.parquet'))
test_df = pd.read_parquet(os.path.join(save_path, 'test_set_processed.parquet'))
val_df = pd.read_parquet(os.path.join(save_path, 'val_set_processed.parquet'))
inference_df = pd.read_parquet(os.path.join(save_path, 'inference_data_latest.parquet'))

# Load the Feature List
with open(os.path.join(save_path, 'feature_list.json'), 'r') as f:
    features = json.load(f)

# Load artifacts
scaler = joblib.load(f'{save_path}feature_scaler.joblib')

with open(f'{save_path}feature_list.json', 'r') as f:
    features = json.load(f)

with open(f'{save_path}class_weights.json', 'r') as f:
    all_class_weights = json.load(f)

print("✅ Data loaded successfully!")
print(f"   Training:   {len(train_df):,} rows")
print(f"   Validation: {len(val_df):,} rows")
print(f"   Test:       {len(test_df):,} rows")
print(f"   Features:   {len(features)}")
print(f"   Crime types with weights: {list(all_class_weights.keys())}")


✅ Data loaded successfully!
   Training:   20,403 rows
   Validation: 4,372 rows
   Test:       4,373 rows
   Features:   45
   Crime types with weights: ['burglary', 'drugs', 'robbery', 'stabbing', 'theft', 'vehicle_theft']


In [113]:
import numpy as np
import pandas as pd
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import make_scorer, f1_score, recall_score, precision_score
import json
import time

### Preparing the data for hyperpramter tuining and training

In [114]:
# Choose the crime type you want to tune
selected_crime = 'vehicle_theft'

target_col = f'target_{selected_crime}_next_week'

# Prepare training data
X_train = train_df[features].copy()
y_train = train_df[target_col].dropna().astype(int)
X_train = X_train.loc[y_train.index]

# Prepare validation data (for final evaluation)
X_val = val_df[features].copy()
y_val = val_df[target_col].dropna().astype(int)
X_val = X_val.loc[y_val.index]

# Get class weights for this crime
crime_weights = all_class_weights[selected_crime]['class_weights']
class_weights = {0: float(crime_weights['0']), 1: float(crime_weights['1'])}

X_test = test_df[features].values
y_test = test_df[target_col].values

# 5. Inference data (Current week for live prediction)
X_inference = inference_df[features].values


## LightGBM (Gradient Boosting Machine)

In [115]:
!pip install lightgbm
import matplotlib.pyplot as plt
import seaborn as sns
from lightgbm import LGBMClassifier

#Hyperparameter tuining

2 Step - Start with RandomSearchCV

and finish with GridSerchCV

- Test with F1-Score or AUC


##Phase 1

### RandomSearchCV

In [116]:
from sklearn.model_selection import RandomizedSearchCV, PredefinedSplit, GridSearchCV
from lightgbm import LGBMClassifier
import lightgbm as lgb

### Initialize the Model with Class Weights

In [117]:
#Initialize the model
lgbm_model = lgb.LGBMClassifier(
    objective='binary',
    class_weight=class_weights,
    random_state=42,
    n_jobs=-1,
    verbosity=-1

)

#### Define the hyperprameters search space

In [118]:
#Define the parameter grid
param_grid = {
    # Higher min_child_samples reduces 'false alarms' (improves Precision)
    'min_child_samples': [20, 30, 50],

    # Moderate leaves prevent the model from becoming too complex
    'num_leaves': [20, 31, 45],

    # Lower n_estimators prevents the model from over-learning noise
    'n_estimators': [500, 600, 800],

    # Keeping learning rate stable
    'learning_rate': [0.01, 0.03],

    # Restricting features slightly to force the model to find the strongest signals
    'feature_fraction': [0.7, 0.8],

    # Standard depth for crime data
    'max_depth': [5, 12, 8, 10, -1],

    'lambda_l1': [0.1, 0.5, 1.0],  # Prunes weak features

    'min_child_samples': [50, 100, 200],# Higher = less overfitting

    'feature_fraction': [0.6, 0.7] # Use fewer features per tree
}


### Setup the Scorer and Search

In [119]:
# Use F1-score to balance Precision and Recall
f1_scorer = make_scorer(f1_score)

# Setup RandomizedSearchCV
random_search = RandomizedSearchCV(
    estimator=lgb.LGBMClassifier(objective='binary', class_weight=class_weights, random_state=42),
    param_distributions= param_grid,
    n_iter=20,
    scoring='f1',
    cv=3,
    verbose=1
)

### View Results

In [120]:
# Fit the search on your prepared data
print(f"--- Tuning hyperparameters for: {selected_crime.upper()} ---")
random_search.fit(X_train, y_train)

# Display the best parameters
print("\n✅ Best Parameters Found:")
best_random_params = random_search.best_params_



--- Tuning hyperparameters for: VEHICLE_THEFT ---
Fitting 3 folds for each of 20 candidates, totalling 60 fits

✅ Best Parameters Found:


In [121]:
# 1. Get predictions (Classes 0 or 1) and probabilities
y_pred = random_search.predict(X_val)
y_probs = random_search.predict_proba(X_val)[:, 1] # Probability of "Crime" (class 1)

# 2. Calculate key metrics
auc_roc = roc_auc_score(y_val, y_probs)
precision = precision_score(y_val, y_pred)
recall = recall_score(y_val, y_pred)

print(f"--- Final Evaluation for: {selected_crime.upper()} ---")
print(f"AUC-ROC Score: {auc_roc:.4f}")
print(f"Precision:    {precision:.4f}")
print(f"Recall:       {recall:.4f}")
print("\nFull Classification Report:")
print(classification_report(y_val, y_pred))

--- Final Evaluation for: VEHICLE_THEFT ---
AUC-ROC Score: 0.9211
Precision:    0.3391
Recall:       0.6842

Full Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.98      0.99      4315
           1       0.34      0.68      0.45        57

    accuracy                           0.98      4372
   macro avg       0.67      0.83      0.72      4372
weighted avg       0.99      0.98      0.98      4372



### Grid Search

In [122]:
# 3. Narrow Grid based on Random Search results
param_grid = {
    'num_leaves': [
        max(20, best_random_params['num_leaves'] - 3),
        best_random_params['num_leaves'],
        min(100, best_random_params['num_leaves'] + 3)
    ],

    # Vary learning_rate around best × [0.8, 1.0, 1.2]
    'learning_rate': [
        round(best_random_params['learning_rate'] * 0.8, 3),
        best_random_params['learning_rate'],
        round(best_random_params['learning_rate'] * 1.2, 3)
    ],

    # Vary n_estimators around best ± 100
    'n_estimators': [
        max(100, best_random_params['n_estimators'] - 100),
        best_random_params['n_estimators'],
        best_random_params['n_estimators'] + 100
    ],

    # Keep these fixed (already optimal from RandomSearch)
    'max_depth': [best_random_params['max_depth']],
    'min_child_samples': [best_random_params['min_child_samples']],
    'feature_fraction': [best_random_params['feature_fraction']],
    'lambda_l1': [best_random_params['lambda_l1']]
}

print("\nGridSearch Parameter Space:")
total_combinations = 1
for param, values in param_grid.items():
    total_combinations *= len(values)
    print(f"  {param:20s}: {values}")

print(f"\n  Total combinations to try: {total_combinations}")
print(f"  (GridSearch will try ALL of them exhaustively)")




GridSearch Parameter Space:
  num_leaves          : [42, 45, 48]
  learning_rate       : [0.008, 0.01, 0.012]
  n_estimators        : [400, 500, 600]
  max_depth           : [-1]
  min_child_samples   : [50]
  feature_fraction    : [0.7]
  lambda_l1           : [0.5]

  Total combinations to try: 27
  (GridSearch will try ALL of them exhaustively)


In [123]:
# 4. Run Exhaustive Grid Search
grid_search = GridSearchCV(
    estimator=lgb.LGBMClassifier(objective='binary', class_weight=class_weights, random_state=42, verbosity = -1),
    param_grid=param_grid,
    scoring='f1',
    cv=3,
    verbose=1
)

grid_search.fit(X_train, y_train)
print("Final Best Parameters:", grid_search.best_params_)

Fitting 3 folds for each of 27 candidates, totalling 81 fits
Final Best Parameters: {'feature_fraction': 0.7, 'lambda_l1': 0.5, 'learning_rate': 0.01, 'max_depth': -1, 'min_child_samples': 50, 'n_estimators': 400, 'num_leaves': 48}


In [124]:
# STEP 3: Compare RandomSearch vs GridSearch

print("\n" + "="*80)
print("COMPARISON: RANDOMSEARCH vs GRIDSEARCH")
print("="*80)

best_grid_params = grid_search.best_params_
best_grid_cv_score = grid_search.best_score_

print(f"\nCross-Validation F1-Score:")
print(f"  RandomSearch: {random_search.best_score_:.4f}")
print(f"  GridSearch:   {best_grid_cv_score:.4f}")
print(f"  Improvement:  {best_grid_cv_score - random_search.best_score_:+.4f}")

print(f"\nParameter Changes:")
print(f"  {'Parameter':<20s} {'RandomSearch':>15s} {'→':^5s} {'GridSearch':>15s}")
print(f"  {'-'*20} {'-'*15} {'-'*5} {'-'*15}")

for param in best_random_params.keys():
    random_val = best_random_params[param]
    grid_val = best_grid_params.get(param, "N/A")
    changed = "✓" if random_val != grid_val else ""
    print(f"  {param:<20s} {str(random_val):>15s} {changed:^5s} {str(grid_val):>15s}")



COMPARISON: RANDOMSEARCH vs GRIDSEARCH

Cross-Validation F1-Score:
  RandomSearch: 0.3003
  GridSearch:   0.3061
  Improvement:  +0.0058

Parameter Changes:
  Parameter               RandomSearch   →        GridSearch
  -------------------- --------------- ----- ---------------
  num_leaves                        45   ✓                48
  n_estimators                     500   ✓               400
  min_child_samples                 50                    50
  max_depth                         -1                    -1
  learning_rate                   0.01                  0.01
  lambda_l1                        0.5                   0.5
  feature_fraction                 0.7                   0.7


In [125]:
# ============================================================================
# STEP 4: Evaluate GridSearch on Validation Set
# ============================================================================

print("\n" + "-"*80)
print("Validation Set Performance")
print("-"*80)

# Get predictions from both models
y_val_pred_random = random_search.predict(X_val)
y_val_proba_random = random_search.predict_proba(X_val)[:, 1]

y_val_pred_grid = grid_search.predict(X_val)
y_val_proba_grid = grid_search.predict_proba(X_val)[:, 1]

# Calculate metrics for RandomSearch
random_precision = precision_score(y_val, y_val_pred_random)
random_recall = recall_score(y_val, y_val_pred_random)
random_f1 = f1_score(y_val, y_val_pred_random)
random_auc = roc_auc_score(y_val, y_val_proba_random)

# Calculate metrics for GridSearch
grid_precision = precision_score(y_val, y_val_pred_grid)
grid_recall = recall_score(y_val, y_val_pred_grid)
grid_f1 = f1_score(y_val, y_val_pred_grid)
grid_auc = roc_auc_score(y_val, y_val_proba_grid)

# Display comparison
print(f"\n{'Metric':<15s} {'RandomSearch':>15s} {'GridSearch':>15s} {'Change':>12s}")
print("-" * 60)
print(f"{'Precision':<15s} {random_precision:>15.4f} {grid_precision:>15.4f} {grid_precision-random_precision:>+12.4f}")
print(f"{'Recall':<15s} {random_recall:>15.4f} {grid_recall:>15.4f} {grid_recall-random_recall:>+12.4f}")
print(f"{'F1-Score':<15s} {random_f1:>15.4f} {grid_f1:>15.4f} {grid_f1-random_f1:>+12.4f}")
print(f"{'AUC-ROC':<15s} {random_auc:>15.4f} {grid_auc:>15.4f} {grid_auc-random_auc:>+12.4f}")



--------------------------------------------------------------------------------
Validation Set Performance
--------------------------------------------------------------------------------

Metric             RandomSearch      GridSearch       Change
------------------------------------------------------------
Precision                0.3391          0.2393      -0.0999
Recall                   0.6842          0.6842      +0.0000
F1-Score                 0.4535          0.3545      -0.0989
AUC-ROC                  0.9211          0.9227      +0.0017


### Select the model

In [126]:
import joblib

# 1. Directly grab the model that is already trained on the best parameters
final_model = grid_search.best_estimator_


### Evaluation

In [127]:
# 1. Predict on the TEST set (the data you held out at the very beginning)
y_test_pred = final_model.predict(X_test)
y_test_probs = final_model.predict_proba(X_test)[:, 1]

# 2. Final Evaluation
test_auc = roc_auc_score(y_test, y_test_probs)

print("="*30)
print("FINAL TEST SET RESULTS (UNSEEN DATA)")
print("="*30)
print(f"Test AUC-ROC: {test_auc:.4f}")
print("\nFinal Classification Report:")
print(classification_report(y_test, y_test_pred))

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


FINAL TEST SET RESULTS (UNSEEN DATA)
Test AUC-ROC: 0.9619

Final Classification Report:
              precision    recall  f1-score   support

         0.0       1.00      0.97      0.98      4317
         1.0       0.26      0.80      0.39        56

    accuracy                           0.97      4373
   macro avg       0.63      0.89      0.69      4373
weighted avg       0.99      0.97      0.98      4373



###Save model

In [131]:
# Save it

model_filename = f'model_{selected_crime}.pkl'
joblib.dump(final_model, model_filename)

print(f"✅ Success! Optimized model for {selected_crime} saved as {model_filename}")

feature_names = X_train.columns.tolist()
joblib.dump(feature_names, f'{selected_crime}_features.pkl')



✅ Success! Optimized model for vehicle_theft saved as model_vehicle_theft.pkl


['vehicle_theft_features.pkl']

- save and Dowload model(for each crime type)
- Get predictions from the model
- backend

In [133]:
#Dowload the model

from google.colab import files
files.download('model_vehicle_theft.pkl')
files.download('vehicle_theft_features.pkl')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [130]:
!ls

burglary_features.pkl  model_drugs.pkl	   robbery_features.pkl
drive		       model_robbery.pkl   sample_data
drugs_features.pkl     model_stabbing.pkl  stabbing_features.pkl
model_burglary.pkl     model_theft.pkl	   theft_features.pkl
